# Federated Analytics

Federated Analytics (FA) computes aggregate statistics — count, mean, variance, histograms, ... — over datasets that stay on remote nodes. No model is trained and no raw data leaves a node: each node computes sufficient statistics locally and the researcher only receives the aggregates.

## Prerequisites

Configure and start two nodes, each holding a dataset registered under the same tag:

```bash
fedbiomed node -p my-first-node dataset add     # then: my-second-node
fedbiomed node -p my-first-node start           # then: my-second-node
```

A node is ready once it logs `Starting task manager`.

## 1. Tabular data

For tabular datasets, statistics are computed per column. Here two nodes hold CSV datasets registered under the `tabular` tag. `Experiment` discovers them and exposes federated analytics through `exp.analytics`.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment

exp = Experiment(tags=['tabular'])

In [ ]:
# Dataset metadata reported by the nodes (no raw data is exposed).
exp.training_data().data()

### Fetching and aggregating

`fetch_stats` sends a request to every node and returns an `FAResult` (results are cached, so repeated calls are not re-requested).

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids        # nodes that contributed

In [ ]:
result.schema          # structure of the result, without values

In [ ]:
result.node_stats()    # raw per-node statistics

In [ ]:
# available_stats(): present in the node replies.
# computable_stats(): additionally derivable locally without re-querying.
print(result.available_stats())
print(result.computable_stats())

In [ ]:
# Aggregate across nodes (count-weighted). Omit the name for every computable stat.
result.global_stats("mean")

In [ ]:
# Shorthand for fetch_stats + global_stats. Served from cache here.
exp.analytics.mean()

### Requesting other statistics

Statistic names are validated before any node is contacted. Parameterised statistics (e.g. histograms) are requested with `fetch_stats_with_args`, but histograms are not yet activated on the nodes, so the request below is rejected.

In [ ]:
# Unknown statistics are rejected up front.
try:
    exp.analytics.fetch_stats(["min", "max"])
except Exception as e:
    print(e)

In [ ]:
# Histograms are requested with explicit bin edges via fetch_stats_with_args,
# but they are not yet activated on the nodes, so this request is rejected.
try:
    result = exp.analytics.fetch_stats_with_args(
        stats_args={"price": {"histogram": {"bin_edges": [100, 125, 150, 175, 200, 250]}}}
    )
except Exception as e:
    print(e)

## 2. Image data (MNIST)

Image datasets are not yet supported by federated analytics (the image dataset type is not active). The request still succeeds but returns an empty result rather than raising — this is unrelated to columns.

In [ ]:
exp = Experiment(tags=['#MNIST', '#dataset'])
exp.training_data().data()

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
# Empty: image datasets are not supported by analytics yet (not a column issue).
print(result.schema)
print(result.available_stats())

## 3. Medical imaging folder (IXI)

The IXI dataset mixes image modalities (`T1`, `T2`, `label`) with a demographics CSV. Statistics are computed only on the named demographic columns; image modalities appear in the schema but yield no statistics.

In [ ]:
exp = Experiment(tags=['ixi'])
exp.training_data().data()

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
# 'demographics' has named sub-columns; image modalities have none.
result.schema

In [ ]:
result.node_stats()

Non-numeric demographics (`DOB`, `STUDY_DATE`, `SITE_NAME`) return `nan` with `count: 0`; image modality keys (`T1`, `T2`, `label`) are empty.

## 4. Secure aggregation

By default the researcher can see each node's contribution before aggregating. With **secure aggregation** every node masks its values, so only the *aggregate* can be recovered — individual contributions are never revealed.

Enable it with `secagg=True`; the analytics API is unchanged. Secure aggregation requires at least 2 nodes.

In [ ]:
exp = Experiment(tags=['tabular'], secagg=True)
print("Secure aggregation active:", exp.secagg.active)

The call is identical to the plaintext case, but the per-node values are no longer recoverable: only the aggregate can be reconstructed. `node_ids` still reports the real participating nodes, while `node_stats()` falls back to the global aggregate.

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids        # real participating nodes (per-node values still hidden)

In [ ]:
result.node_stats()    # no per-node values: logs an info message and returns the global aggregate

In [ ]:
result.global_stats("mean")

### Variance and standard deviation

`variance` and `std` need two passes (global mean, then the centered sum of squares). Each pass runs as its own secure-aggregation round — transparently under `secagg=True`.

In [ ]:
exp.analytics.variance()

### Choosing the scheme: LOM vs Joye-Libert

Two secure-aggregation schemes are available; both produce the **same** statistics and differ only in the cryptographic protocol and cost.

| | **LOM** (default) | **Joye-Libert** |
|---|---|---|
| Approach | Pairwise masking (Diffie-Hellman) | Additive homomorphic encryption |
| Per-round setup | None — keys reused across rounds | Server key + biprime |
| Speed | Faster, lighter | Heavier setup |

`secagg=True` selects LOM. To use Joye-Libert, pass an explicit `SecureAggregation`.

In [ ]:
from fedbiomed.researcher.secagg import SecureAggregation
from fedbiomed.common.constants import SecureAggregationSchemes

exp_jls = Experiment(
    tags=['tabular'],
    secagg=SecureAggregation(scheme=SecureAggregationSchemes.JOYE_LIBERT),
)
print("Scheme:", exp_jls.secagg.scheme)

In [ ]:
# Same API, same result as LOM — only the protocol underneath differs.
exp_jls.analytics.fetch_stats("mean").global_stats("mean")